## **Goal**

Build a weighted semantic retrieval system over a reproducible Semantic Scholar corpus.

### **Input**


    Semantic Scholar latest release (pinned)

    abstracts dataset → text

    papers dataset → metadata (title, year, venue, etc.)

    User query (string)


### **Output**

    Top-10 most relevant papers

    Weighted score per paper

    Field-level contribution breakdown


## **Steps**

    Resolve latest release and inspect datasets

    Sample papers from abstracts (text-first)

    Join metadata from papers using corpusid

    Clean + prepare fields (title, abstract, metadata)

    Embed fields

    Apply heuristic weighted scoring (query-aware)

    Retrieve and display top results

## **Data Formatting**

This notebook assumes input data has already been scraped and joined into a flat CSV.

Required columns:
- `corpusid`
- `title`
- `abstract`

Optional columns:
- `year`
- `venue`
- `authors`
- `s2fieldsofstudy`
- `citationcount`
- `referencecount`

This notebook starts from that CSV and handles:
1. loading
2. cleaning
3. field construction
4. embedding
5. weighted retrieval

In [3]:
# ----------------------------
# Load joined paper dataset
# ----------------------------

import pandas as pd

CSV_PATH = "large_joined_sample.csv"   # swap later for larger runs (dev_joined_sample is a 119 paper toy dev set)

df = pd.read_csv(CSV_PATH)

print(df.shape)
print(df.columns.tolist())
print(df[["corpusid", "title", "abstract"]].head(3))

(1500, 10)
['corpusid', 'abstract', 'title', 'year', 'venue', 'authors', 's2fieldsofstudy', 'citationcount', 'referencecount', 'url']
    corpusid                                              title  \
0   55551962  Facilitating nurses' knowledge of the utilisat...   
1  218493897  Towards Optimal Search: An Efficient Search Al...   
2   10880698  THE PROTOPLANETARY DISKS IN THE NEARBY MASSIVE...   

                                            abstract  
0  An integrative literature review of identified...  
1  Search techniques are an integral part for tex...  
2  The formation of stars in massive clusters is ...  


In [4]:
# ----------------------------
# Normalize expected columns
# ----------------------------

REQUIRED_COLS = ["corpusid", "title", "abstract"]
OPTIONAL_COLS = [
    "year", "venue", "authors", "s2fieldsofstudy",
    "citationcount", "referencecount"
]

for col in REQUIRED_COLS:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

for col in OPTIONAL_COLS:
    if col not in df.columns:
        df[col] = ""

df["corpusid"] = df["corpusid"].astype(str)
df["title"] = df["title"].fillna("").astype(str)
df["abstract"] = df["abstract"].fillna("").astype(str)
df["venue"] = df["venue"].fillna("").astype(str)

# keep rows with actual retrieval text
df = df[(df["title"].str.len() > 0) | (df["abstract"].str.len() > 0)].reset_index(drop=True)

print(df.shape)

(1500, 10)


In [5]:
# ----------------------------
# Text cleaning
# ----------------------------

import re
import ast

def clean_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def parse_maybe_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    x = str(x).strip()
    if not x:
        return []
    try:
        val = ast.literal_eval(x)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    return [x]

In [6]:
# ----------------------------
# Build retrievable fields
# ----------------------------

def fields_of_study_to_text(fos):
    fos_list = parse_maybe_list(fos)

    names = []
    for item in fos_list:
        if isinstance(item, dict):
            if "category" in item:
                names.append(str(item["category"]))
            elif "name" in item:
                names.append(str(item["name"]))
            else:
                names.append(str(item))
        else:
            names.append(str(item))

    return ", ".join(names)

def authors_to_text(authors):
    author_list = parse_maybe_list(authors)

    names = []
    for item in author_list:
        if isinstance(item, dict) and "name" in item:
            names.append(str(item["name"]))
        else:
            names.append(str(item))

    return ", ".join(names)

def build_fields(row):
    title = clean_text(row["title"])
    abstract = clean_text(row["abstract"])
    venue = clean_text(row.get("venue", ""))
    fos_text = clean_text(fields_of_study_to_text(row.get("s2fieldsofstudy", "")))
    author_text = clean_text(authors_to_text(row.get("authors", "")))

    metadata_text = ". ".join(
        x for x in [
            f"venue: {venue}" if venue else "",
            f"fields of study: {fos_text}" if fos_text else "",
            f"authors: {author_text}" if author_text else "",
        ]
        if x
    )

    return {
        "title": title,
        "abstract": abstract,
        "metadata": metadata_text,
    }

papers = []

for _, row in df.iterrows():
    papers.append({
        "corpusid": row["corpusid"],
        "year": row.get("year", ""),
        "citationcount": row.get("citationcount", 0),
        "referencecount": row.get("referencecount", 0),
        "url": row.get("url", ""),
        "fields": build_fields(row)
    })

print(len(papers))
print(papers[0]["fields"])

1500
{'title': "Facilitating nurses' knowledge of the utilisation of reflexology in adults with chronic diseases to enable informed health education during comprehensive nursing care", 'abstract': 'An integrative literature review of identified scientific evidence, published from January 2000 to December 2008, of the utilisation of reflexology as complementary and alternative medicine (CAM) modalities to promote well-being and quality of life in adults with chronic diseases was done to facilitate nurses to give informed health education during comprehensive nursing care to patients with chronic diseases. Selected accessible databases were searched purposefully for research articles ( N = 1171). Pre-set inclusion criteria were applied during the study selection process. The methodological study quality was reviewed and appraised with appropriate tools from the Critical Appraisal Skills Programme (CASP) and the American Dietetic Association’s (ADA) Evidence analysis manual ( n = 21). Evi

# **Embeddings**

In [7]:
# ----------------------------
# Load embedding model
# ----------------------------

import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"

model = SentenceTransformer(
    MODEL_NAME,
    trust_remote_code=True
)

print("loaded:", MODEL_NAME)

/scratch/mmd9604/conda-envs/asr/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<All keys matched successfully>


loaded: nomic-ai/nomic-embed-text-v1.5


In [8]:
# ----------------------------
# Embedding helpers
# ----------------------------

def embed_texts(texts, model, prefix, batch_size=32):
    texts = [clean_text(t) for t in texts]
    texts = [f"{prefix} {t}" if t else "" for t in texts]

    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    return np.asarray(embs)

def embed_query(query, model):
    emb = model.encode(
        [f"search_query: {clean_text(query)}"],
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    return np.asarray(emb[0])

def cosine_score(a, b):
    return float(np.dot(a, b))

In [9]:
# ----------------------------
# Embed fields
# ----------------------------

FIELD_NAMES = ["title", "abstract", "metadata"]

for paper in papers:
    paper["embeddings"] = {}

for field in FIELD_NAMES:
    texts = [p["fields"].get(field, "") for p in papers]
    embs = embed_texts(texts, model, prefix="search_document:")

    for i, paper in enumerate(papers):
        text = texts[i]
        paper["embeddings"][field] = embs[i] if text else None

print("done")
for field, emb in papers[0]["embeddings"].items():
    print(field, "->", None if emb is None else emb.shape)

Batches: 100%|██████████| 47/47 [00:01<00:00, 28.07it/s]

done
title -> (768,)
abstract -> (768,)
metadata -> (768,)


In [10]:
# ----------------------------
# Heuristic query-aware weights
# ----------------------------

BASE_WEIGHTS = {
    "title": 0.30,
    "abstract": 0.55,
    "metadata": 0.15,
}

METHOD_WORDS = {
    "method", "methods", "approach", "algorithm", "model", "framework", "architecture"
}

VENUE_WORDS = {
    "conference", "journal", "venue", "workshop", "published"
}

AUTHOR_WORDS = {
    "author", "authors", "by", "written"
}

FIELD_WORDS = {
    "field", "topic", "area", "domain", "study"
}

def get_query_weights(query):
    q = clean_text(query).lower()
    words = set(q.split())

    weights = dict(BASE_WEIGHTS)

    if words & METHOD_WORDS:
        weights["abstract"] += 0.05

    if words & VENUE_WORDS:
        weights["metadata"] += 0.10

    if words & AUTHOR_WORDS:
        weights["metadata"] += 0.10

    if words & FIELD_WORDS:
        weights["metadata"] += 0.05

    total = sum(weights.values())
    weights = {k: v / total for k, v in weights.items()}
    return weights

In [11]:
# ----------------------------
# Retrieval
# ----------------------------

def score_paper(query_emb, paper, weights):
    active_fields = []
    active_weights = []

    for field, weight in weights.items():
        emb = paper["embeddings"].get(field)
        if emb is not None:
            active_fields.append(field)
            active_weights.append(weight)

    if not active_fields:
        return -1.0, {}, {}

    weight_sum = sum(active_weights)
    active_weights = [w / weight_sum for w in active_weights]

    raw_scores = {}
    weighted_contribs = {}
    total_score = 0.0

    for field, w in zip(active_fields, active_weights):
        s = cosine_score(query_emb, paper["embeddings"][field])
        raw_scores[field] = s
        weighted_contribs[field] = w * s
        total_score += w * s

    return total_score, raw_scores, weighted_contribs


def retrieve_top_k(query, papers, model, k=10):
    query_emb = embed_query(query, model)
    weights = get_query_weights(query)

    results = []

    for paper in papers:
        total_score, raw_scores, weighted_contribs = score_paper(query_emb, paper, weights)
        results.append({
            "corpusid": paper["corpusid"],
            "score": total_score,
            "weights": weights,
            "raw_scores": raw_scores,
            "weighted_contribs": weighted_contribs,
            "title": paper["fields"].get("title", ""),
            "abstract": paper["fields"].get("abstract", ""),
            "metadata": paper["fields"].get("metadata", ""),
            "url": paper.get("url", ""),
            "year": paper.get("year", ""),
            "citationcount": paper.get("citationcount", 0),
        })

    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:k]

# **Test Query**

In [15]:
query = ""

results = retrieve_top_k(query, papers, model, k=10)

print("query weights:", results[0]["weights"])
print()

for i, r in enumerate(results, 1):
    print(f"{i}. {r['corpusid']}  score={r['score']:.4f}  year={r['year']}")
    print("   title:", r["title"][:160])

    sorted_contribs = sorted(r["weighted_contribs"].items(), key=lambda x: x[1], reverse=True)
    print("   weighted contribs:", [(k, round(v, 4)) for k, v in sorted_contribs])

    if r["url"]:
        print("   url:", r["url"])

    print()

query weights: {'title': 0.3, 'abstract': 0.55, 'metadata': 0.15}

1. 268948822  score=0.7143  year=2024.0
   title: Human Attention Restoration, Flow, and Creativity: A Conceptual Integration
   weighted contribs: [('abstract', 0.3927), ('title', 0.2196), ('metadata', 0.102)]
   url: https://www.semanticscholar.org/paper/240ac60d925293cc70db1e96d7f33684979b58cd

2. 204376191  score=0.7127  year=2019.0
   title: Strains of Dissent: Popular Music and Everyday Resistance in WWII France, 1940–1945
   weighted contribs: [('abstract', 0.3999), ('title', 0.2076), ('metadata', 0.1051)]
   url: https://www.semanticscholar.org/paper/3675ef5e82369ef1148481aa6bcd26096a24473d

3. 253014395  score=0.7114  year=2022.0
   title: Gambaran penggunaan antikoagulan pada pasien ST-Elevatıon Myocardıal Infarctıon (STEMI)
   weighted contribs: [('abstract', 0.3975), ('title', 0.2123), ('metadata', 0.1016)]
   url: https://www.semanticscholar.org/paper/5ba8e65f51001ae8cee1efcaa30c459e82f51da6

4. 1221  score

# **Eval**

In [17]:
# ----------------------------
# Config for submission export
# ----------------------------

import json
import re
from pathlib import Path

RUN_ID = "massimo_weighted_semantic_v1"
BENCHMARK_PATH = "benchmark_queries.jsonl"      # teammate's file
SUBMISSION_PATH = f"{RUN_ID}.jsonl"

In [18]:
# ----------------------------
# Load benchmark queries
# ----------------------------

benchmark_queries = []
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            benchmark_queries.append(json.loads(line))

print("num queries:", len(benchmark_queries))
print(benchmark_queries[:2])

num queries: 100
[{'queryId': 'q_001', 'query': 'machine learning for healthcare', 'year': None, 'fields': 'paperId,title,abstract,authors,year,publicationDate,fieldsOfStudy,s2FieldsOfStudy,url'}, {'queryId': 'q_002', 'query': 'graph learning in biology', 'year': None, 'fields': 'paperId,title,abstract,authors,year,publicationDate,fieldsOfStudy,s2FieldsOfStudy,url'}]


In [19]:
# ----------------------------
# Helper: extract Semantic Scholar paperId from URL
# ----------------------------

def extract_paper_id_from_url(url: str) -> str:
    """
    Semantic Scholar URLs usually look like:
    https://www.semanticscholar.org/paper/<paperId>
    or
    https://www.semanticscholar.org/paper/<slug>/<paperId>
    """
    url = str(url).strip()
    if not url:
        return ""

    m = re.search(r"/paper/(?:[^/]+/)?([0-9a-f]{40})$", url)
    if m:
        return m.group(1)

    # fallback: last path component if it looks like a hex id
    last = url.rstrip("/").split("/")[-1]
    if re.fullmatch(r"[0-9a-f]{40}", last):
        return last

    return ""

In [20]:
# ----------------------------
# Optional: year filter helper
# ----------------------------

def parse_year_filter(year_value):
    """
    Supports:
    - None / null
    - exact year like '2021'
    - lower-bounded like '2023-'
    - range like '2019-2022'
    """
    if year_value is None:
        return None

    y = str(year_value).strip()
    if not y or y.lower() == "null":
        return None

    if re.fullmatch(r"\d{4}", y):
        year = int(y)
        return ("exact", year)

    m = re.fullmatch(r"(\d{4})-", y)
    if m:
        return ("min", int(m.group(1)))

    m = re.fullmatch(r"(\d{4})-(\d{4})", y)
    if m:
        return ("range", int(m.group(1)), int(m.group(2)))

    return None


def year_matches_filter(paper_year, year_filter):
    if year_filter is None:
        return True

    try:
        py = int(float(paper_year))
    except Exception:
        return False

    kind = year_filter[0]
    if kind == "exact":
        return py == year_filter[1]
    if kind == "min":
        return py >= year_filter[1]
    if kind == "range":
        return year_filter[1] <= py <= year_filter[2]

    return True

In [21]:
# ----------------------------
# Retrieval wrapper for evaluator format
# ----------------------------

def retrieve_top_k_for_eval(query, papers, model, k=10, year=None):
    """
    Same retriever, but:
    - enforces optional year filter
    - returns only results with valid paperId
    """
    query_emb = embed_query(query, model)
    weights = get_query_weights(query)
    year_filter = parse_year_filter(year)

    results = []

    for paper in papers:
        if not year_matches_filter(paper.get("year", None), year_filter):
            continue

        total_score, raw_scores, weighted_contribs = score_paper(query_emb, paper, weights)
        paper_id = extract_paper_id_from_url(paper.get("url", ""))

        if not paper_id:
            continue

        results.append({
            "paperId": paper_id,
            "score": float(total_score),
            "title": paper["fields"].get("title", ""),
            "year": paper.get("year", None),
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    # dedupe by paperId just in case
    deduped = []
    seen = set()
    for r in results:
        if r["paperId"] not in seen:
            deduped.append(r)
            seen.add(r["paperId"])
        if len(deduped) >= k:
            break

    return deduped

In [22]:
# ----------------------------
# Build submission JSONL
# ----------------------------

submission_rows = []

for q in benchmark_queries:
    query_id = q["queryId"]
    query_text = q["query"]
    year_value = q.get("year", None)

    top_results = retrieve_top_k_for_eval(
        query=query_text,
        papers=papers,
        model=model,
        k=10,
        year=year_value,
    )

    row = {
        "runId": RUN_ID,
        "queryId": query_id,
        "results": [
            {
                "rank": rank,
                "paperId": r["paperId"],
                "score": r["score"],
            }
            for rank, r in enumerate(top_results, start=1)
        ]
    }

    submission_rows.append(row)

print("built rows:", len(submission_rows))
print(submission_rows[0])

built rows: 100
{'runId': 'massimo_weighted_semantic_v1', 'queryId': 'q_001', 'results': [{'rank': 1, 'paperId': 'ad4087c451f843a47e1cbb73f1f2588c0f02a392', 'score': 0.6781540602445603}, {'rank': 2, 'paperId': 'bfc88f8733cb6d2f14eae9218055384716fbb814', 'score': 0.6663091719150543}, {'rank': 3, 'paperId': '98939483dedc2c60b2be7cd879f95e5d47371b12', 'score': 0.6659971654415131}, {'rank': 4, 'paperId': '49723d3d1c595d2b360d0c574a739e05a5110572', 'score': 0.6571428269147873}, {'rank': 5, 'paperId': '21457fc8c312554ce3353c5d46c778bfa34890cc', 'score': 0.6345510572195053}, {'rank': 6, 'paperId': '07acb1c3b2e5932e146bbb9c6a8a20a350a04c4b', 'score': 0.6336383700370789}, {'rank': 7, 'paperId': '62297783bcecb41acb110c3cbe03996ed036c77c', 'score': 0.626403197646141}, {'rank': 8, 'paperId': '1c1d7f13cc9b3ec7949f74ab80f24cf5ee78f769', 'score': 0.6263394355773927}, {'rank': 9, 'paperId': 'f4f73e6450f9d112bd57e51525de9c49cae79137', 'score': 0.6259852021932603}, {'rank': 10, 'paperId': '22c03c4a29da1

In [23]:
# ----------------------------
# Validate submission shape
# ----------------------------

def validate_submission_rows(rows):
    for row in rows:
        assert "runId" in row
        assert "queryId" in row
        assert "results" in row
        assert len(row["results"]) == 10, f"{row['queryId']} does not have 10 results"

        ranks = [r["rank"] for r in row["results"]]
        assert ranks == list(range(1, 11)), f"{row['queryId']} ranks invalid: {ranks}"

        paper_ids = [r["paperId"] for r in row["results"]]
        assert all(isinstance(pid, str) and pid for pid in paper_ids), f"{row['queryId']} has bad paperId"
        assert len(set(paper_ids)) == 10, f"{row['queryId']} has duplicate paperIds"

validate_submission_rows(submission_rows)
print("submission format looks valid")

submission format looks valid


In [24]:
# ----------------------------
# Write JSONL submission
# ----------------------------

with open(SUBMISSION_PATH, "w", encoding="utf-8") as f:
    for row in submission_rows:
        f.write(json.dumps(row) + "\n")

print("wrote:", SUBMISSION_PATH)

wrote: massimo_weighted_semantic_v1.jsonl


In [25]:
# ----------------------------
# Quick reload check
# ----------------------------

loaded_rows = []
with open(SUBMISSION_PATH, "r", encoding="utf-8") as f:
    for line in f:
        loaded_rows.append(json.loads(line))

print("reloaded rows:", len(loaded_rows))
print(loaded_rows[0])

reloaded rows: 100
{'runId': 'massimo_weighted_semantic_v1', 'queryId': 'q_001', 'results': [{'rank': 1, 'paperId': 'ad4087c451f843a47e1cbb73f1f2588c0f02a392', 'score': 0.6781540602445603}, {'rank': 2, 'paperId': 'bfc88f8733cb6d2f14eae9218055384716fbb814', 'score': 0.6663091719150543}, {'rank': 3, 'paperId': '98939483dedc2c60b2be7cd879f95e5d47371b12', 'score': 0.6659971654415131}, {'rank': 4, 'paperId': '49723d3d1c595d2b360d0c574a739e05a5110572', 'score': 0.6571428269147873}, {'rank': 5, 'paperId': '21457fc8c312554ce3353c5d46c778bfa34890cc', 'score': 0.6345510572195053}, {'rank': 6, 'paperId': '07acb1c3b2e5932e146bbb9c6a8a20a350a04c4b', 'score': 0.6336383700370789}, {'rank': 7, 'paperId': '62297783bcecb41acb110c3cbe03996ed036c77c', 'score': 0.626403197646141}, {'rank': 8, 'paperId': '1c1d7f13cc9b3ec7949f74ab80f24cf5ee78f769', 'score': 0.6263394355773927}, {'rank': 9, 'paperId': 'f4f73e6450f9d112bd57e51525de9c49cae79137', 'score': 0.6259852021932603}, {'rank': 10, 'paperId': '22c03c4a29